## 편향과 분산(Bias & Variance)

- 편향이 낮은 모델은 항상 편향이 높은 모델보다 좋지는 않음
    - 편향이 높은 모델은 너무 간단해서 주어진 데이터의 관계를 잘 학습하지 못함
    - 편향이 낮은 모델은 주어진데이터의 관계를 잘 학습함
- 분산은 다양한 테스트데이터가 주어졌을 때 모델의 성능이 얼마나 일관적인지 확인

## 정규화

- 머신 러닝 모델을 학습시킬때 θ값들이 너무 커지는 것을 방지해주는 기법
- L1 : Lasso 모델, L2 : Ridge 모델
- L1 은 θ의 절댓값을 더고
- L2 는 θ의 제곱을 더해줌

In [2]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error


In [3]:
admission_df = pd.read_csv('admission.csv')
admission_df.head()


,Serial No.,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
0,1,337,118,4,4.5,4.5,9.65,1,0.92
1,2,324,107,4,4.0,4.5,8.87,1,0.76
2,3,316,104,3,3.0,3.5,8.00,1,0.72
3,4,322,110,3,3.5,2.5,8.67,1,0.80
4,5,314,103,2,2.0,3.0,8.21,0,0.65


## X와 y 나누기

In [4]:
X = admission_df.drop(columns=['Chance of Admit'])
y = admission_df['Chance of Admit']


## 6차 다항 회귀 모델을 이용해 정규화하기

In [5]:
poly = PolynomialFeatures(6)
X_poly = poly.fit_transform(X)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_poly, y, test_size=0.3, random_state=42
)


In [7]:
model = LinearRegression()
model.fit(X_train, y_train)

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

print("Polynomial Regression (No Regularization)")
print("Train MSE:", mean_squared_error(y_train, train_pred))
print("Test  MSE:", mean_squared_error(y_test, test_pred))


Polynomial Regression (No Regularization)
Train MSE: 1.4863253609724547e-18
Test  MSE: 0.6171067135338145


#### 결과 해석

training MSE는 매우 작은데 상대적으로 Test MSE는 상대적으로 큼. = 모델이 train 데이터에 과적합


## L1 정규화

In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [9]:
lasso = Lasso(alpha=0.001, max_iter=2000)
lasso.fit(X_train_scaled, y_train)

train_pred = lasso.predict(X_train_scaled)
test_pred = lasso.predict(X_test_scaled)

print("\nPolynomial Regression + Lasso (L1)")
print("Train MSE:", mean_squared_error(y_train, train_pred))
print("Test  MSE:", mean_squared_error(y_test, test_pred))



Polynomial Regression + Lasso (L1)
Train MSE: 0.0030787804415102317
Test  MSE: 0.003293486061920808


#### 결과해석

처음에 비하면 test MSE가 줄어듬. 두 값의 차이가 줄어듦으로써 과적합이 완화된 걸 알 수 있음.

## L2 정규화

In [10]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

train_pred = ridge.predict(X_train_scaled)
test_pred = ridge.predict(X_test_scaled)

print("\nPolynomial Regression + Ridge (L2)")
print("Train MSE:", mean_squared_error(y_train, train_pred))
print("Test  MSE:", mean_squared_error(y_test, test_pred))



Polynomial Regression + Ridge (L2)
Train MSE: 0.0019441519721206588
Test  MSE: 0.005274227248110787


#### 결과해석

이번에서 두 사이의 차이가 작아 과적합 완화 확인

=> 정리해 보면, 높은 차수의 다항 회귀는 과소적합을 막는 데는 효과적이지만 과적합이 발생하기 쉬움. 여기에 Ridge나 Lasso 같은 정규화를 적용하면, training 성능은 조금 떨어지더라도 test 성능이 개선되면서 더 좋은 모델을 만들 수 있음.